# Exercise 5: A particle on an ellipse

**Block 2: Twiss, ellipses and emittance**

**Learning objectives**
* Track one particle for many turns and see that its turn-by-turn points lie on an ellipse.
* Compute the **Courant–Snyder invariant** $\varepsilon_i = \gamma x^2 + 2\alpha x p_x + \beta p_x^2$ and check that it is constant.
* See that the ellipse changes shape around the ring, but its area ($\pi\varepsilon_i$) does not.

**Motivation.** If you look at a particle once per turn, at one place in the ring, it jumps
around in the $(x, p_x)$ plane. But the jumps are not random: all points lie on one ellipse.
The size of that ellipse is the single most important number describing that particle.

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from beamlib import elements as E, optics as O, lattices as L, tracking as T, plotting as P
from beamlib.plotting import COLORS, MM
P.setup()

ring = L.fodo_ring()                     # the FODO ring of Exercise 4 (no sextupoles: linear)
one_turn = T.make_one_turn_map(ring)     # function: bunch -> bunch after one turn
twA = O.twiss_from_matrix(O.one_turn_matrix(ring)[:2, :2])
print(f"Location A (ring start, centre of QF): beta = {twA['beta']:.3f} m, alpha = {round(twA['alpha'], 3) + 0.0:.3f}")

## 1. Track one particle

A bunch is an array of shape (4, N) with rows $(x, p_x, y, p_y)$. Here N = 1.
`T.track(bunch, one_turn, n_turns)` returns an object whose `.x` and `.px` have shape
(n_turns + 1, N): one row per turn.

**Predict first.** The particle starts at $x$ = 2 mm with $p_x$ = 0. Sketch what you expect to see
after 500 turns if you plot ($x$, $p_x$) once per turn. Where will the largest $x$ be?

*Your prediction:*

In [ ]:
p0 = np.array([[2e-3], [0.0], [0.0], [0.0]])
res = T.track(p0, one_turn, 500)
x, px = res.x[:, 0], res.px[:, 0]

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(x * MM, px * MM, s=4, c=np.arange(len(x)), cmap="viridis")
for k in range(6):
    ax.annotate(str(k), (x[k] * MM, px[k] * MM), fontsize=9, color="k",
                xytext=(4, 4), textcoords="offset points")
ax.set_xlabel(r"$x$ [mm]"); ax.set_ylabel(r"$p_x$ [mrad]"); ax.set_title("500 turns at location A (numbers: first turns)")
plt.show()

## 2. The Courant–Snyder invariant

**Task.** Implement `cs_invariant(x, px, beta, alpha)`. It must work on whole arrays.
Then check that it is the same on every turn.

In [ ]:
def cs_invariant(x, px, beta: float, alpha: float):
    """Single-particle emittance eps_i = gamma x^2 + 2 alpha x px + beta px^2 [m rad]."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
eps_i = cs_invariant(x, px, twA["beta"], twA["alpha"])
print(f"eps_i = {eps_i[0] * 1e6:.6f} mm mrad, relative spread over 500 turns: {np.ptp(eps_i) / eps_i.mean():.1e}")
assert np.allclose(eps_i, eps_i[0], rtol=1e-10), "eps_i should be constant turn after turn"
print("GOOD! The Courant-Snyder invariant is conserved")

The ellipse with this $\varepsilon_i$ has its largest $x$ at $\sqrt{\varepsilon_i\beta}$ and its largest $p_x$ at $\sqrt{\varepsilon_i\gamma}$.
Below we draw it on top of the tracked points (`P.plot_ellipse`).

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(x * MM, px * MM, s=4, color=COLORS["x"], label="tracked")
P.plot_ellipse(ax, eps_i[0], twA["beta"], twA["alpha"], ls="--", lw=1, label=r"$\varepsilon_i$ ellipse")
x_max = np.sqrt(eps_i[0] * twA["beta"])
ax.axvline(x_max * MM, color=COLORS["grey"], ls=":"); ax.axvline(-x_max * MM, color=COLORS["grey"], ls=":")
ax.set_xlabel(r"$x$ [mm]"); ax.set_ylabel(r"$p_x$ [mrad]"); ax.legend()
plt.show()
assert np.isclose(np.abs(x).max(), x_max, rtol=1e-3)

## 3. Nested ellipses

Particles with different starting amplitudes sit on different, nested ellipses with the **same shape**.
The shape is set by the lattice $(\beta, \alpha)$; the size by the particle ($\varepsilon_i$).

In [ ]:
amps = np.array([0.5, 1.0, 2.0, 3.0]) * 1e-3
bunch = np.zeros((4, len(amps))); bunch[0, :] = amps
res4 = T.track(bunch, one_turn, 300)
fig, ax = plt.subplots(figsize=(5, 5))
for j, c in enumerate(P.amplitude_colors(len(amps))):
    ax.scatter(res4.x[:, j] * MM, res4.px[:, j] * MM, s=3, color=c, label=rf"$x_0$ = {amps[j] * MM} mm")
ax.set_xlabel(r"$x$ [mm]"); ax.set_ylabel(r"$p_x$ [mrad]"); ax.legend(fontsize=8)
plt.show()
ratios = cs_invariant(res4.x[0], res4.px[0], twA["beta"], twA["alpha"]) / amps**2
print("eps_i / x0^2 =", ratios.round(5), " (= gamma, the same for all: eps_i grows like amplitude^2)")

## 4. Another place in the ring

Now look at the *same particle* at location B: the entrance of the first dipole, 0.7 m after the
ring start. To do this we "rotate" the ring so that it starts at B: `ring[2:] + ring[:2]`.
We also need to move the particle from A to B through the first two elements.

**Predict first.** At B, will the ellipse be upright like at A, or tilted? Bigger or smaller area?

*Your prediction:*

In [ ]:
ringB = ring[2:] + ring[:2]
one_turn_B = T.make_one_turn_map(ringB)
twB = O.twiss_from_matrix(O.one_turn_matrix(ringB)[:2, :2])
print(f"Location B: beta = {twB['beta']:.3f} m, alpha = {twB['alpha']:.3f}")

pB = p0
for el in ring[:2]:          # move the particle from A to B
    pB = el.track(pB)
resB = T.track(pB, one_turn_B, 500)
xB, pxB = resB.x[:, 0], resB.px[:, 0]
eps_B = cs_invariant(xB, pxB, twB["beta"], twB["alpha"])
print(f"eps_i at A = {eps_i[0] * 1e6:.6f} mm mrad, at B = {eps_B[0] * 1e6:.6f} mm mrad")
assert np.allclose(eps_B, eps_i[0], rtol=1e-10), "the invariant is the same everywhere in the ring"
print("GOOD! Same particle, same eps_i at A and at B")

In [ ]:
amps_B = np.array([0.5, 1.0, 2.0, 3.0]) * 1e-3
bunch_B = np.zeros((4, len(amps_B))); bunch_B[0, :] = amps_B
res5 = T.track(bunch_B, one_turn_B, 300)
fig, ax = plt.subplots(figsize=(5, 5))
for j, c in enumerate(P.amplitude_colors(len(amps_B))):
    ax.scatter(res5.x[:, j] * MM, res5.px[:, j] * MM, s=3, color=c, label=rf"$x_0$ = {amps[j] * MM} mm")
ax.set_xlabel(r"$x$ [mm]"); ax.set_ylabel(r"$p_x$ [mrad]"); ax.legend(fontsize=8)
plt.show()

## 5. Area of the ellipse

**Task.** Measure the area enclosed by the tracked points directly, without using any Twiss
parameter: sort the points by their angle around the origin (`np.arctan2(px, x)`) and apply the
*shoelace formula* for the area of a polygon,
area = $\frac{1}{2} |\sum_k (x_k p_{k+1} − x_{k+1} p_k)|$.
Compare area / $\pi$ with $\varepsilon_i$ at both locations.

In [ ]:
def polygon_area(x: np.ndarray, px: np.ndarray) -> float:
    """Area of the closed curve through the points, sorted by angle around (0, 0)."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
area_A, area_B = polygon_area(x, px), polygon_area(xB, pxB)
print(f"area/pi at A: {area_A / np.pi * 1e6:.5f} mm mrad, at B: {area_B / np.pi * 1e6:.5f} mm mrad, "
      f"eps_i = {eps_i[0] * 1e6:.5f} mm mrad")
assert np.isclose(area_A / np.pi, eps_i[0], rtol=1e-3) and np.isclose(area_B / np.pi, eps_i[0], rtol=1e-3)
print("GOOD! area = pi * eps_i at both places")

fig, axs = plt.subplots(1, 2, figsize=(11, 5))
for ax, (xx, pp, tw, lab) in zip(axs, [(x, px, twA, "A (QF centre)"), (xB, pxB, twB, "B (dipole entrance)")]):
    ax.scatter(xx * MM, pp * MM, s=6, color=COLORS["x"])
    P.plot_ellipse(ax, eps_i[0], tw["beta"], tw["alpha"], ls="--", lw=1)
    ax.set_title(rf"location {lab}: $\beta$ = {tw['beta']:.2f} m, $\alpha$ = {round(tw['alpha'], 2) + 0.0:.2f}")
    ax.set_xlabel(r"$x$ [mm]"); ax.set_ylabel(r"$p_x$ [mrad]")
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-0.7, 0.7)
plt.show()

## Reflection questions

1. At location A the ellipse is upright, at B it is tilted. What does the sign of $\alpha$ tell you
   about the beam at B (is it getting bigger or smaller as it moves on)?
2. Why is $\varepsilon_i$ the same at A and B even though $x_\text{max}$ is different?
3. If you started the particle with twice the amplitude, by how much would $\varepsilon_i$ change? And the area?